# **KKBOX Churn Prediction And Retention Intelligence System - Data Preparation and Feature Engineering**

## **1. Objectives**

This notebook prepares and transforms the raw KKBOX datasets into a clean, structured, and modeling‑ready table for churn prediction.

The goal is to:

- Perform light cleaning and sanity checks across all raw data sources

- Merge transactional, demographic, and behavioral logs into a unified customer‑level dataset

- Handle missing values, invalid entries, and inconsistent data types

- Engineer meaningful features such as tenure, engagement groups, recency, and payment variability

- Reduce noise through outlier capping and low‑frequency category grouping

- Prevent data leakage and ensure modeling integrity

At this stage, the focus is on data quality, structure, and feature creation.
No modeling is performed yet — the output is a clean, enriched dataset ready for machine learning.

## **2. Load Data & Basic Setup**

In [143]:

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


current_dir = os.getcwd()


while not os.path.exists(os.path.join(current_dir, "data", "raw")):
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        raise FileNotFoundError("Project root with data/raw not found")
    current_dir = parent


os.chdir(current_dir)

print("Project root:", os.getcwd())
print("data/raw exists:", os.path.exists("data/raw"))


pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

target = "is_churn"

Project root: c:\Users\pauli\OneDrive\Documentos\GitHub\customer-churn-intelligence-system
data/raw exists: True


## **3. Light Cleaning**

In [144]:
# 3.1. Train dataset

train = pd.read_csv("data/raw/train_v2.csv")
train = train.drop_duplicates(subset="msno")

train["is_churn"] = train["is_churn"].astype(int)

In [145]:
# 3.2 Members dataset
members = pd.read_csv("data/raw/members_v3.csv")

members = members.drop_duplicates(subset="msno")

members.loc[(members["bd"] < 0) | (members["bd"] > 100), "bd"] = np.nan

members["gender"] = members["gender"].astype("category")

members["registration_init_time"] = pd.to_datetime(
    members["registration_init_time"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

members["city"] = pd.to_numeric(members["city"], errors="coerce")
members["registered_via"] = pd.to_numeric(members["registered_via"], errors="coerce")


In [146]:
# 3.3 Transactions dataset
transactions = pd.read_csv("data/raw/transactions_v2.csv")

transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

transactions["membership_expire_date"] = pd.to_datetime(
    transactions["membership_expire_date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

transactions = transactions[
    transactions["transaction_date"] <= transactions["membership_expire_date"]
]

num_cols = [
    "payment_method_id", "payment_plan_days", "plan_list_price",
    "actual_amount_paid", "is_auto_renew", "is_cancel"
]

transactions[num_cols] = transactions[num_cols].apply(pd.to_numeric, errors="coerce")

transactions.loc[transactions["actual_amount_paid"] < 0, "actual_amount_paid"] = np.nan
transactions.loc[transactions["plan_list_price"] < 0, "plan_list_price"] = np.nan
transactions.loc[transactions["payment_plan_days"] <= 0, "payment_plan_days"] = np.nan


In [147]:
# 3.4 User Logs dataset
user_logs = pd.read_csv("data/raw/user_logs_v2.csv")

log_cols = [
    "num_25", "num_50", "num_75", "num_985",
    "num_100", "num_unq", "total_secs"
]

user_logs[log_cols] = user_logs[log_cols].apply(pd.to_numeric, errors="coerce")

user_logs.loc[user_logs["total_secs"] < 0, "total_secs"] = np.nan
user_logs.loc[user_logs["num_unq"] < 0, "num_unq"] = np.nan

user_logs["date"] = pd.to_datetime(
    user_logs["date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

user_logs = user_logs[
    (user_logs["date"] >= pd.Timestamp("2010-01-01")) &
    (user_logs["date"] <= pd.Timestamp("2020-01-01"))
]


In [148]:
# 3.5 Quick Sanity Checks

print("Train:", train.shape)
print("Members:", members.shape)
print("Transactions:", transactions.shape)
print("User logs:", user_logs.shape)

print("Missing values in train:\n", train.isnull().mean().sort_values(ascending=False).head(10))
print("Missing values in members:\n", members.isnull().mean().sort_values(ascending=False).head(10))
print("Missing values in transactions:\n", transactions.isnull().mean().sort_values(ascending=False).head(10))
print("Missing values in user_logs:\n", user_logs.isnull().mean().sort_values(ascending=False).head(10))


print("Duplicate users in train:", train["msno"].duplicated().sum())
print("Duplicate users in members:", members["msno"].duplicated().sum())
print("Exact duplicate rows in transactions:", transactions.duplicated().sum())
print("Exact duplicate rows in user_logs:", user_logs.duplicated().sum())

train = train.reset_index(drop=True)
members = members.reset_index(drop=True)
transactions = transactions.reset_index(drop=True)
user_logs = user_logs.reset_index(drop=True)

Train: (970960, 2)
Members: (6769473, 6)
Transactions: (1425903, 9)
User logs: (18396362, 9)
Missing values in train:
 msno       0
is_churn   0
dtype: float64
Missing values in members:
 gender                   1
bd                       0
city                     0
msno                     0
registered_via           0
registration_init_time   0
dtype: float64
Missing values in transactions:
 payment_plan_days        0
msno                     0
payment_method_id        0
plan_list_price          0
actual_amount_paid       0
is_auto_renew            0
transaction_date         0
membership_expire_date   0
is_cancel                0
dtype: float64
Missing values in user_logs:
 msno         0
date         0
num_25       0
num_50       0
num_75       0
num_985      0
num_100      0
num_unq      0
total_secs   0
dtype: float64
Duplicate users in train: 0
Duplicate users in members: 0
Exact duplicate rows in transactions: 0
Exact duplicate rows in user_logs: 0


## **4. Aggregate Transactions & Usaer Logs**

In [149]:
# 4.1  Aggregate Transactions (from transaction-level to user-level)

# Aggregate transactions to customer level
transactions_agg = transactions.groupby("msno").agg({
    "actual_amount_paid": ["sum", "mean"],
    "payment_plan_days": ["sum", "mean"],
    "plan_list_price": "mean",
    "is_auto_renew": ["mean", "max"],
    "is_cancel": ["mean", "max"],
    "transaction_date": ["count", "min", "max"],
    "membership_expire_date": "max"
}).reset_index()

# Flatten multi-index column names
transactions_agg.columns = [
    "msno",
    "total_amount_paid",
    "avg_amount_paid",
    "total_payment_plan_days",
    "avg_payment_plan_days",
    "avg_plan_list_price",
    "auto_renew_rate",
    "has_auto_renew",
    "cancel_rate",
    "has_cancelled",
    "transaction_count",
    "first_transaction_date",
    "last_transaction_date",
    "membership_expire_date"
]

print("Transactions aggregated:", transactions_agg.shape)
transactions_agg.head()

Transactions aggregated: (1194686, 14)


,msno,total_amount_paid,avg_amount_paid,total_payment_plan_days,avg_payment_plan_days,avg_plan_list_price,auto_renew_rate,has_auto_renew,cancel_rate,has_cancelled,transaction_count,first_transaction_date,last_transaction_date,membership_expire_date
0,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,"1,599","1,599",395,395,"1,599",0,0,0,0,1,2016-10-23,2016-10-23,2018-02-06
1,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,99,99,30,30,99,1,1,0,0,1,2017-03-15,2017-03-15,2017-04-15
2,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,298,149,60,30,149,1,1,0,0,2,2017-02-28,2017-03-31,2017-05-19
3,+++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc=,149,149,30,30,149,1,1,0,0,1,2017-03-26,2017-03-26,2017-04-26
4,++/9R3sX37CjxbY/AaGvbwr3QkwElKBCtSvVzhCBDOk=,149,149,30,30,149,1,1,0,0,1,2017-03-15,2017-03-15,2017-04-15


In [150]:
# 4.2 Aggregate User Logs (from activity-level to user-level)

user_logs_agg = user_logs.groupby("msno").agg({
    "total_secs": "sum",
    "num_unq": "sum"
}).reset_index()

print("User logs aggregated:", user_logs_agg.shape)
user_logs_agg.head()


User logs aggregated: (1103894, 3)


,msno,total_secs,num_unq
0,+++IZseRRiQS9aaSkH6cMYU6bGDcxUieAi/tH67sC5s=,"117,907",530
1,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,"192,528",885
2,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,"115,411",468
3,+++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc=,"149,897",828
4,++/9R3sX37CjxbY/AaGvbwr3QkwElKBCtSvVzhCBDOk=,"116,433",230


## **5. Merge**

In [151]:
# 5.1 Merge all datasets into a single modeling table

df = (
    train
    .merge(members, on="msno", how="left")
    .merge(transactions_agg, on="msno", how="left")
    .merge(user_logs_agg, on="msno", how="left")
)

print("Final dataset shape:", df.shape)
df.head()

Final dataset shape: (970960, 22)


,msno,is_churn,city,bd,gender,registered_via,registration_init_time,total_amount_paid,avg_amount_paid,total_payment_plan_days,avg_payment_plan_days,avg_plan_list_price,auto_renew_rate,has_auto_renew,cancel_rate,has_cancelled,transaction_count,first_transaction_date,last_transaction_date,membership_expire_date,total_secs,num_unq
0,ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=,1,5,28,male,3,2013-12-23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaT,"80,599",348
1,f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=,1,13,20,male,3,2013-12-23,180,180,30,30,180,0,0,0,0,1,2017-03-11,2017-03-11,2017-04-11,"6,987",30
2,zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=,1,13,18,male,3,2013-12-27,300,150,150,75,150,0,0,0,0,2,2017-03-11,2017-03-14,2017-06-15,"67,810",432
3,8iF/+8HY8lJKFrTc7iR9ZYGCG2Ecrogbc2Vy5YhsfhQ=,1,1,0,NaN,7,2014-01-09,"1,490",149,300,30,149,1,1,0,0,10,2015-08-08,2015-12-08,2018-01-08,NaN,NaN
4,K6fja4+jmoZ5xG6BypqX80Uw/XKpMgrEMdG2edFOxnA=,1,13,35,female,7,2014-01-25,792,99,240,30,99,1,1,0,1,8,2016-10-01,2017-03-16,2017-09-18,"239,882",548


In [152]:
# 5.2 Quick Sanity Check after merge

print("Missing values after merge:")
print(df.isnull().mean().sort_values(ascending=False).head(20))

print("Duplicate users in final df:", df["msno"].duplicated().sum())



Missing values after merge:
gender                    1
num_unq                   0
total_secs                0
bd                        0
city                      0
registered_via            0
registration_init_time    0
avg_payment_plan_days     0
has_auto_renew            0
auto_renew_rate           0
total_payment_plan_days   0
avg_plan_list_price       0
avg_amount_paid           0
total_amount_paid         0
first_transaction_date    0
transaction_count         0
cancel_rate               0
has_cancelled             0
membership_expire_date    0
last_transaction_date     0
dtype: float64
Duplicate users in final df: 0


In [153]:
# 5.3 Save raw merged data

df.to_csv("data/interim/df_raw_merged.csv", index=False)
print("Merged dataset saved to interim.")

Merged dataset saved to interim.


## **6. Deep Cleaning**

### **6.1 Load Merged Dataset**

In [154]:
current_dir = os.getcwd()


while not os.path.exists(os.path.join(current_dir, "data", "interim")):
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        raise FileNotFoundError("Project root with data/interim not found")
    current_dir = parent


os.chdir(current_dir)

print("Project root:", os.getcwd())
print("data/interim exists:", os.path.exists("data/interim"))


pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

target = "is_churn"

Project root: c:\Users\pauli\OneDrive\Documentos\GitHub\customer-churn-intelligence-system
data/interim exists: True


### **6.2. Validate Data Structure (Sanity Checks)**

In [155]:
# df raw merged overview
df = pd.read_csv("data/interim/df_raw_merged.csv")

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 970960 entries, 0 to 970959
Data columns (total 22 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   msno                     970960 non-null  object 
 1   is_churn                 970960 non-null  int64  
 2   city                     860967 non-null  float64
 3   bd                       860444 non-null  float64
 4   gender                   388905 non-null  object 
 5   registered_via           860967 non-null  float64
 6   registration_init_time   860967 non-null  object 
 7   total_amount_paid        931409 non-null  float64
 8   avg_amount_paid          931409 non-null  float64
 9   total_payment_plan_days  931409 non-null  float64
 10  avg_payment_plan_days    931408 non-null  float64
 11  avg_plan_list_price      931409 non-null  float64
 12  auto_renew_rate          931409 non-null  float64
 13  has_auto_renew           931409 non-null  float64
 14  canc

,msno,is_churn,city,bd,gender,registered_via,registration_init_time,total_amount_paid,avg_amount_paid,total_payment_plan_days,avg_payment_plan_days,avg_plan_list_price,auto_renew_rate,has_auto_renew,cancel_rate,has_cancelled,transaction_count,first_transaction_date,last_transaction_date,membership_expire_date,total_secs,num_unq
0,ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=,1,5,28,male,3,2013-12-23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"80,599",348
1,f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=,1,13,20,male,3,2013-12-23,180,180,30,30,180,0,0,0,0,1,2017-03-11,2017-03-11,2017-04-11,"6,987",30
2,zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=,1,13,18,male,3,2013-12-27,300,150,150,75,150,0,0,0,0,2,2017-03-11,2017-03-14,2017-06-15,"67,810",432
3,8iF/+8HY8lJKFrTc7iR9ZYGCG2Ecrogbc2Vy5YhsfhQ=,1,1,0,NaN,7,2014-01-09,"1,490",149,300,30,149,1,1,0,0,10,2015-08-08,2015-12-08,2018-01-08,NaN,NaN
4,K6fja4+jmoZ5xG6BypqX80Uw/XKpMgrEMdG2edFOxnA=,1,13,35,female,7,2014-01-25,792,99,240,30,99,1,1,0,1,8,2016-10-01,2017-03-16,2017-09-18,"239,882",548


### **6.3. Rename Column (bd to Age)**

In [156]:
df = df.rename(columns={"bd": "age"})

### **6.4. Missing Value Strategy**

In [157]:
# 6.4.1 Demographics
# Invalid ages → NaN
df.loc[(df["age"] <= 0) | (df["age"] > 100), "age"] = np.nan

# Gender missing → "unknown"
df["gender"] = df["gender"].fillna("unknown")

In [158]:
# 6.4.2 Engagement
# Missing logs = no activity → meaningful
df["total_secs"] = df["total_secs"].fillna(0)
df["num_unq"] = df["num_unq"].fillna(0)

In [159]:
# 6.4.3 Transactions
transaction_cols = [
    "total_amount_paid", "avg_amount_paid",
    "total_payment_plan_days", "avg_payment_plan_days",
    "avg_plan_list_price", "auto_renew_rate",
    "has_auto_renew", "cancel_rate", "has_cancelled",
    "transaction_count"
]

df[transaction_cols] = df[transaction_cols].fillna(0)

### **6.5. Fix Data Types**

In [160]:
date_cols = [
    "registration_init_time",
    "first_transaction_date",
    "last_transaction_date",
    "membership_expire_date"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

### **6.6. Statistical Distribution Checks**

In [161]:
pd.set_option("display.float_format", "{:,.0f}".format)
df[["avg_amount_paid", "total_secs"]].describe(percentiles=[.99])

,avg_amount_paid,total_secs
count,"970,960","970,960"
mean,138,"108,629"
std,135,"173,586"
min,0,0
50%,149,"49,982"
99%,600,"850,075"
max,"2,000","14,337,391"


### **6.7. Outlier Treatment (Capping)**

In [162]:
# We cap only to stabilize models
cap_pay = df["avg_amount_paid"].quantile(0.99)
df["avg_amount_paid"] = np.where(df["avg_amount_paid"] > cap_pay, cap_pay, df["avg_amount_paid"])

cap_secs = df["total_secs"].quantile(0.99)
df["total_secs"] = np.where(df["total_secs"] > cap_secs, cap_secs, df["total_secs"])

### **6.8. Low-Frequency Category Grouping**

In [163]:
# 6.8.1 City
city_freq = df["city"].value_counts(normalize=True)
rare_cities = city_freq[city_freq < 0.01].index
df["city_grouped"] = df["city"].replace(rare_cities, "other")

In [164]:
# 6.8.2 Registered Via
via_freq = df["registered_via"].value_counts(normalize=True)
rare_via = via_freq[via_freq < 0.01].index
df["registered_via_grouped"] = df["registered_via"].replace(rare_via, "other")


## **7. Feature Engineering**

In [165]:
# 7.1 Tenure
df["customer_tenure_days"] = (
    df["last_transaction_date"] - df["registration_init_time"]
).dt.days.fillna(0)

In [166]:
# 7.2 Engagement Buckets
df["listening_group"] = pd.qcut(
    df["total_secs"],
    q=4,
    labels=["Low", "Medium-Low", "Medium-High", "High"]
)

In [167]:
# 7.3 Payment Variability
df["payment_variability"] = df["total_amount_paid"] / (df["transaction_count"] + 1)


In [168]:
# 7.4 Recency (days since last transaction)
cutoff_date = df["last_transaction_date"].max()
df["days_since_last_transaction"] = (cutoff_date - df["last_transaction_date"]).dt.days
df["days_since_last_transaction"] = df["days_since_last_transaction"].fillna(df["days_since_last_transaction"].max())

## **8. Multicollinearity Check** 

In [169]:
df.corr(numeric_only=True)["is_churn"].sort_values(ascending=False)

is_churn                       1
days_since_last_transaction    1
cancel_rate                    0
has_cancelled                  0
total_payment_plan_days        0
payment_variability            0
avg_payment_plan_days          0
avg_plan_list_price            0
total_amount_paid              0
avg_amount_paid                0
city                           0
transaction_count              0
num_unq                       -0
total_secs                    -0
registered_via                -0
customer_tenure_days          -0
age                           -0
has_auto_renew                -0
auto_renew_rate               -0
Name: is_churn, dtype: float64

## **9. Leakage Prevention**

In [170]:
leakage_cols = [
    "membership_expire_date",  # future expiry
    "last_transaction_date"    # may occur after churn
]

df = df.drop(columns=leakage_cols)

## **10. Final Cleaned Dataset**

In [172]:
final_cols = [
    "msno", "is_churn",
    "gender", "age", "city_grouped", "registered_via_grouped",
    "avg_amount_paid", "total_amount_paid",
    "has_auto_renew", "has_cancelled",
    "total_secs", "num_unq",
    "customer_tenure_days", "listening_group",
    "payment_variability"
]

df_clean = df[final_cols].copy()
df_clean.head()


,msno,is_churn,gender,age,city_grouped,registered_via_grouped,avg_amount_paid,total_amount_paid,has_auto_renew,has_cancelled,total_secs,num_unq,customer_tenure_days,listening_group,payment_variability
0,ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=,1,male,28,5,3,0,0,0,0,"80,599",348,0,Medium-High,0
1,f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=,1,male,20,13,3,180,180,0,0,"6,987",30,"1,174",Medium-Low,90
2,zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=,1,male,18,13,3,150,300,0,0,"67,810",432,"1,173",Medium-High,100
3,8iF/+8HY8lJKFrTc7iR9ZYGCG2Ecrogbc2Vy5YhsfhQ=,1,unknown,NaN,1,7,149,"1,490",1,0,0,0,698,Low,135
4,K6fja4+jmoZ5xG6BypqX80Uw/XKpMgrEMdG2edFOxnA=,1,female,35,13,7,99,792,1,1,"239,882",548,"1,146",High,88


## **11. Save Clean Dataset**

In [173]:
df_clean.to_csv("data/processed/df_clean.csv", index=False)
print("Clean dataset saved to processed.")

Clean dataset saved to processed.
